# Assignment 1

### Group Members

### Installation (Only Run if Needed)

In [ ]:
%pip install pymongo

### Import Packages

In [1]:
from pymongo import MongoClient, ReadPreference
from pymongo.write_concern import WriteConcern
from pymongo.read_concern import ReadConcern

import pymongo
import time

### Default Connections

In [ ]:
uri = "mongodb://localhost:27017/"
client = MongoClient(uri)

In [ ]:
db = client["university"]
collection = db["students"]

In [ ]:
# Define the query filter
query = {"student_id": "S007"}

# Retrieve the first matching document
result = collection.find_one(query)
result

### Replica Set Connection

In [2]:
client = MongoClient(
    "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208",
    serverSelectionTimeoutMS=5000,
    w="majority",
    readConcernLevel="majority"
)

In [3]:
hello = client.admin.command("hello")

print("Replica set:", hello.get("setName"))
print("Primary:", hello.get("primary"))
print("Writable primary:", hello.get("isWritablePrimary"))

Replica set: dsa5208
Primary: mongo1:27017
Writable primary: True


In [4]:
for name in ["mongo1", "mongo2", "mongo3"]:
    try:
        c = MongoClient(f"mongodb://{name}:27017/?directConnection=true", 
                        serverSelectionTimeoutMS=5000)
        info = c.admin.command("hello")
        print(name, "is connected to:", info.get("me"))
    except Exception as e:
        print(name, "FAILED:", e)

mongo1 is connected to: mongo1:27017
mongo2 is connected to: mongo2:27017
mongo3 is connected to: mongo3:27017


In [5]:
db = client.get_database('university', 
                         read_preference=ReadPreference.SECONDARY
                        )

#### Read Follow Writes

In [6]:
students = db["students"]

'''
for i in range(999):
    students.insert_one({
            "student_id": f"S{i:03}",
            "student_name": f"Robert Tan - {i}",
            "student_status": "Graduate",
            "senority" : f'Year {i%4}'
            "academic_load": "Full Time"
    })
'''

students.insert_one({
        "student_id": "S001",
        "student_name": "Robert Tan",
        "student_status": "Undergraduate",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6a9bd02a2ecac588047deb36'), acknowledged=True)

In [7]:
db.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bcffdfd199549cfd4031a'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [8]:
client3 = MongoClient(
    "mongodb://mongo3:27017/?directConnection=true",
    serverSelectionTimeoutMS=3000
)

In [9]:
db3 = client3.get_database('university')

In [10]:
db3.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bcffdfd199549cfd4031a'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [11]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 38,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788596266, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788596266, 1)}

In [14]:
db.students.update_one(
   { "student_id": "S001"},         
   { "$set" : {"student_status": "Graduate"} }
)

db.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bcffdfd199549cfd4031a'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [15]:
db3.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bcffdfd199549cfd4031a'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [19]:
status = client.admin.command("replSetGetStatus")

for m in status["members"]:
    print(m["name"], m["stateStr"], m.get("optimeDate"))

mongo1:27017 PRIMARY 2026-09-05 08:18:12
mongo2:27017 SECONDARY 2026-09-05 08:18:12
mongo3:27017 SECONDARY 2026-09-05 08:17:46


In [20]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

db3.students.find_one({"student_id": "S001"})

{'_id': ObjectId('6a9bcffdfd199549cfd4031a'),
 'student_id': 'S001',
 'student_name': 'Robert Tan',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [21]:
with client.start_session(causal_consistency=True) as session:
    db.students.update_one(
        {"student_id": "S001"},
        {"$set": {"student_status": "Post Doc"}},
        session=session
    )
    
    doc = db.students.find_one({"student_id": "S001"}, session=session)
    print("With session:", doc)

With session: {'_id': ObjectId('6a9bcffdfd199549cfd4031a'), 'student_id': 'S001', 'student_name': 'Robert Tan', 'student_status': 'Post Doc', 'senority': 'Year 1', 'academic_load': 'Full Time'}


In [22]:
status = client.admin.command("replSetGetStatus")

for m in status["members"]:
    print(m["name"], m["stateStr"], m.get("optimeDate"))

mongo1:27017 PRIMARY 2026-09-05 08:18:23
mongo2:27017 SECONDARY 2026-09-05 08:18:23
mongo3:27017 SECONDARY 2026-09-05 08:18:23


#### Monotonic Reads

In [23]:
students.insert_one({
        "student_id": "S002",
        "student_name": "Feyd Johnston",
        "student_status": "Undergraduate",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6a9bd0572ecac588047deb38'), acknowledged=True)

In [24]:
db.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6a9bd0572ecac588047deb38'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [25]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 40,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788596311, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788596311, 1)}

In [26]:
for i in range(2, 6):
    db.students.update_one(
        {"student_id": "S002"},
        {"$set": {"senority": f"Year {i}"}}
    )

In [27]:
no_session_reads = []
for _ in range(10):
    doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
        {"student_id": "S002"}
    )
    no_session_reads.append(doc["senority"])

print("No session:", no_session_reads)

No session: ['Year 5', 'Year 5', 'Year 5', 'Year 1', 'Year 5', 'Year 1', 'Year 1', 'Year 5', 'Year 5', 'Year 5']


In [33]:
with client.start_session(causal_consistency=True) as session:
    with_session_reads = []
    
    for _ in range(10):
        try:
            doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
                {"student_id": "S002"}, session=session, max_time_ms=3000
            )
            with_session_reads.append(doc["senority"])
        except Exception as e:
            with_session_reads.append("TIMEOUT")

print("With session:", with_session_reads)

With session: ['Year 1', 'Year 5', 'Year 5', 'Year 5', 'Year 5', 'TIMEOUT', 'Year 5', 'TIMEOUT', 'Year 5', 'TIMEOUT']


In [34]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

{'count': 42,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788596452, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788596311, 1)}

#### Monotonic Write

In [35]:
students.insert_one({
        "student_id": "S003",
        "student_name": "Alan White",
        "student_status": "Post Doc",
        "senority" : "Year 1",
        "academic_load": "Full Time"
})

InsertOneResult(ObjectId('6a9bd0ed2ecac588047deb39'), acknowledged=True)

In [36]:
db3.students.find_one({"student_id": "S003"})

{'_id': ObjectId('6a9bd0ed2ecac588047deb39'),
 'student_id': 'S003',
 'student_name': 'Alan White',
 'student_status': 'Post Doc',
 'senority': 'Year 1',
 'academic_load': 'Full Time'}

In [37]:
db.students.update_one(
    {"student_id": "S003"},
    {"$set": {"academic_history": ["B.Sc. Computer Science"]}},
    upsert=True
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788596467, 1), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788596467, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788596467, 1), 'updatedExisting': True}, acknowledged=True)

In [38]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 42,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788596467, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788596467, 1)}

In [39]:
db.students.update_one({"student_id": "S003"}, {"$push": {"academic_history": "M.Sc. Data Science"}})
db.students.update_one({"student_id": "S003"}, {"$push": {"academic_history": "PhD Information Systems"}})

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788596473, 2), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788596473, 2), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788596473, 2), 'updatedExisting': True}, acknowledged=True)

In [40]:
print("mongo3 mid-stall:", db3.students.find_one({"student_id": "S003"})["academic_history"])

mongo3 mid-stall: ['B.Sc. Computer Science']


In [43]:
db.students.find_one({"student_id": "S003"})["academic_history"]

['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']

In [44]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

time.sleep(2)

print("mongo3 (after sync is enabled):", db3.students.find_one({"student_id": "S003"})["academic_history"])

mongo3 (after sync is enabled): ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']


#### Write follow read

In [45]:
courses = db["courses"]

courses.insert_one({
        "course_id": "C001",
        "course_name": "Programming Methodology",
        "course_status": "ACTIVE",
        "vacancy" : "30"
})

InsertOneResult(ObjectId('6a9bd6b22ecac588047deb3a'), acknowledged=True)

In [49]:
db.courses.find_one({"course_id": "C001"})

{'_id': ObjectId('6a9bd6b22ecac588047deb3a'),
 'course_id': 'C001',
 'course_name': 'Programming Methodology',
 'course_status': 'ACTIVE',
 'vacancy': '30'}

In [47]:
db3.courses.find_one({"course_id": "C001"})

{'_id': ObjectId('6a9bd6b22ecac588047deb3a'),
 'course_id': 'C001',
 'course_name': 'Programming Methodology',
 'course_status': 'ACTIVE',
 'vacancy': '30'}

In [50]:
students.update_one(
    {"student_id": "S002"},
    {"$set": {"enrollment_note": None}}
)

UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788598053, 1), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788598053, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788598053, 1), 'updatedExisting': True}, acknowledged=True)

In [51]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "alwaysOn"
})

{'count': 50,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598102, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598102, 1)}

In [53]:
course = db.courses.with_options(read_preference=ReadPreference.SECONDARY).find_one({"course_id": "C001"})
print("Read course ", course["course_name"], " with status:", course["course_status"])

db.students.update_one(
    {"student_id": "S002"},
    {"$set": {"enrollment_note": f"Enrolled in {course['course_name']} while course_status was: {course['course_status']}"}}
)

Read course  Programming Methodology  with status: ACTIVE


UpdateResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1788598208, 1), 't': 3}, 'nModified': 1, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1788598208, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1788598208, 1), 'updatedExisting': True}, acknowledged=True)

In [54]:
db3.courses.find_one({"course_id": "C001"})

{'_id': ObjectId('6a9bd6b22ecac588047deb3a'),
 'course_id': 'C001',
 'course_name': 'Programming Methodology',
 'course_status': 'ACTIVE',
 'vacancy': '30'}

In [55]:
db3.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6a9bd1779f5e71580a600788'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 5',
 'academic_load': 'Full Time',
 'enrollment_note': None}

In [57]:
db.students.find_one({"student_id": "S002"})

{'_id': ObjectId('6a9bd1779f5e71580a600788'),
 'student_id': 'S002',
 'student_name': 'Feyd Johnston',
 'student_status': 'Undergraduate',
 'senority': 'Year 5',
 'academic_load': 'Full Time',
 'enrollment_note': 'Enrolled in Programming Methodology while course_status was: ACTIVE'}

In [61]:
with client.start_session(causal_consistency=True) as session:
    course = db.courses.with_options(read_preference=ReadPreference.SECONDARY).find_one(
        {"course_id": "C001"}, session=session
    )
    print("Session read course status:", course["course_status"])

    db.students.update_one(
        {"student_id": "S002"},
        {"$set": {"enrollment_note": f"Session enrollment, course_status: {course['course_status']}"}},
        session=session
    )

    try:
        doc = db.students.with_options(read_preference=ReadPreference.SECONDARY).find_one(
            {"student_id": "S002"}, session=session, max_time_ms=3000
        )
        print("Session read-back:", doc)
    except Exception as e:
        print("Timed out:", e)

Session read course status: ACTIVE
Session read-back: {'_id': ObjectId('6a9bd1779f5e71580a600788'), 'student_id': 'S002', 'student_name': 'Feyd Johnston', 'student_status': 'Undergraduate', 'senority': 'Year 5', 'academic_load': 'Full Time', 'enrollment_note': 'Session enrollment, course_status: ACTIVE'}


In [62]:
client3.admin.command({
    "configureFailPoint": "rsSyncApplyStop",
    "mode": "off"
})

{'count': 52,
 'ok': 1.0,
 '$clusterTime': {'clusterTime': Timestamp(1788598582, 1),
  'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00',
   'keyId': 0}},
 'operationTime': Timestamp(1788598102, 1)}